<a href="https://colab.research.google.com/github/axyojp/katago-on-colab/blob/main/katago-on-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# はじめに
KataGo を利用する際に GPU を利用したいことがあると思います。
GPU マシンを用意するのも、難しい設定を行うのも大変な場合があるため、Google Colaboratory で動作するようにしました。

単に動かすだけではなく、GUI で碁盤をクリックして操作できるようにしてあります。


基本的には上から下まで順番にセルを実行することで利用できるはずです。




## 事前設定
必ず Runtime の設定で GPU を選択してください。
L4 でも十分です。


![ランタイムのタイプを変更 / Change runtime type](https://raw.githubusercontent.com/axyojp/katago-on-colab/main/img/runtime-setting.png)


In [ ]:
%%bash
nvidia-smi
echo ""
nvcc --version

In [ ]:
# @title KataGo Configuration
# @markdown You can configure the KataGo version and model URL here.

KATAGO_RELEASE_URL = "https://github.com/lightvector/KataGo/releases/download/v1.16.4/katago-v1.16.4-cuda12.1-cudnn8.9.7-linux-x64.zip" # @param {type:"string"}
MODEL_URL = "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-b28c512nbt-s12163512064-d5648012427.bin.gz" # @param {type:"string"}

import os

katago_zip_name = KATAGO_RELEASE_URL.split('/')[-1]
model_file_name = MODEL_URL.split('/')[-1]

print(f"KataGo Release URL: {KATAGO_RELEASE_URL}")
print(f"Model URL: {MODEL_URL}")

In [ ]:
# @title モデルの選択 (Select Model - Optional)
# @markdown katagotraining.org から最新のネットワーク一覧を取得し、ドロップダウンで選択できます。
# @markdown このセルを実行しなくても、上のセルの MODEL_URL がそのまま使われます。
# @markdown 選択した後に「1. インストール」セルを (再) 実行するとダウンロードされます。

import json as _json
import urllib.request as _urlreq
import ipywidgets as _widgets
from IPython.display import display as _display

_NETWORKS_API = "https://katagotraining.org/api/networks/?format=json"

def _fetch_networks(limit=15):
    """最新のネットワーク一覧を (表示ラベル, モデルURL) のリストで返す"""
    # デフォルトの Python User-Agent は API 側にブロックされるため明示する
    req = _urlreq.Request(_NETWORKS_API, headers={"User-Agent": "katago-on-colab/1.0"})
    with _urlreq.urlopen(req, timeout=15) as r:
        data = _json.load(r)
    nets = []
    for n in data["results"][:limit]:
        if not n.get("model_file"):
            continue
        size_mb = (n.get("model_file_bytes") or 0) / 1e6
        label = f'{n["name"]}  [{n.get("network_size", "?")} / {size_mb:.0f}MB / {n["created_at"][:10]}]'
        nets.append((label, n["model_file"]))
    return nets

if "MODEL_URL" not in globals():
    print("先に KataGo Configuration セルを実行してください。")
else:
    try:
        _choices = _fetch_networks()
        # 現在の設定値を先頭に置き、一覧に同じものがあれば除外する
        _options = [(f"現在の設定: {MODEL_URL.split('/')[-1]}", MODEL_URL)]
        _options += [c for c in _choices if c[1] != MODEL_URL]

        _dd = _widgets.Dropdown(options=_options, value=MODEL_URL, description="Model:",
                                layout=_widgets.Layout(width="90%"))
        _out = _widgets.Output()

        def _on_model_change(change):
            global MODEL_URL, model_file_name
            MODEL_URL = change["new"]
            model_file_name = MODEL_URL.split("/")[-1]
            with _out:
                _out.clear_output()
                print(f"MODEL_URL を更新しました: {model_file_name}")
                print("「1. インストール」セルを再実行するとダウンロードされます。")

        _dd.observe(_on_model_change, names="value")
        _display(_dd, _out)
        print(f"最新 {len(_choices)} 件のネットワークを取得しました。現在の選択: {model_file_name}")
    except Exception as e:
        print(f"ネットワーク一覧の取得に失敗しました ({e})")
        print(f"既定の MODEL_URL をそのまま使います: {model_file_name}")


## 1. インストール (Installation)

上で設定したバージョンのKataGoとモデルをダウンロード・インストールします。
左側の再生ボタン（▶）を押して実行してください。

In [ ]:
# @title インストールとセットアップ (Installation & Setup)
# KataGo install セクションの内容を一括実行する

# 1. 依存ライブラリのインストール
!sudo apt-get update
!sudo apt-get install -y libzip4 libgoogle-perftools-dev opencl-headers ocl-icd-libopencl1 ocl-icd-opencl-dev
!sudo apt-get install -y libcudnn8

# 2. KataGo本体のダウンロードと解凍
os.makedirs("katago", exist_ok=True)
!wget -nc {KATAGO_RELEASE_URL}
!unzip -o {katago_zip_name} -d katago
!chmod +x katago/katago

# 3. モデルのダウンロード
!cd katago && wget -nc {MODEL_URL}


## 2. 設定ファイルの作成 (Create Config)

KataGoの動作設定ファイルを作成します。
ここもそのまま実行してください。

In [ ]:
%%writefile katago/gtp.cfg

# Config for KataGo C++ GTP engine
# Colab の L4 ランタイムを基準にした設定 (A100 でもそのまま動作します)

# ===========================================================================
# Logs and files
# ===========================================================================
logDir = gtp_logs
logAllGTPCommunication = true
logSearchInfo = true
logSearchInfoForChosenMove = false
logToStderr = false

# ===========================================================================
# Analysis
# ===========================================================================
analysisPVLen = 15
reportAnalysisWinratesAs = SIDETOMOVE

# ===========================================================================
# Rules
# ===========================================================================
koRule = SIMPLE
scoringRule = TERRITORY
taxRule = SEKI
multiStoneSuicideLegal = false
hasButton = false
whiteHandicapBonus = 0
friendlyPassOk = false

# ===========================================================================
# Bot behavior
# ===========================================================================
allowResignation = true
resignThreshold = -0.90
resignConsecTurns = 3
# 相手番でも先読みする設定 (ポンダリング)。有効にすると GPU が回り続けて
# Colab のコンピューティングユニットを裏で消費するため、無効にしています。
ponderingEnabled = false

# ===========================================================================
# Search limits
# ===========================================================================
# スレッド数。Colab L4 + b28 モデルでの実測 (-tune, 2026-07) では 24 が最良でした。
# GPU を変えた場合は「設定の自動調整」セルの探索結果に合わせて調整してください
numSearchThreads = 24
lagBuffer = 1.0

searchFactorAfterOnePass = 0.50
searchFactorAfterTwoPass = 0.25
searchFactorWhenWinning = 0.40
searchFactorWhenWinningThreshold = 0.95

# ===========================================================================
# GPU settings
# ===========================================================================

# バッチサイズ。256 で L4 / A100 とも安定して動作します。
nnMaxBatchSize = 256

# NNキャッシュサイズ (2のべき乗)。23 = 約838万局面で、L4 ランタイムの
# システムRAMに収まる範囲で読みの重複を減らします。
# A100 (ハイメモリ) を使う場合は 25 まで上げても構いません。
nnCacheSizePowerOfTwo = 23

nnMutexPoolSizePowerOfTwo = 17

# ===========================================================================
# Root move selection and biases
# ===========================================================================
chosenMoveTemperatureEarly = 0.5
chosenMoveTemperatureHalflife = 19
chosenMoveTemperature = 0.10
chosenMoveSubtract = 0
chosenMovePrune = 1
useLcbForSelection = true
lcbStdevs = 5.0
minVisitPropForLCB = 0.15

# ===========================================================================
# Internal params
# ===========================================================================
winLossUtilityFactor = 1.0
staticScoreUtilityFactor = 0.10
dynamicScoreUtilityFactor = 0.30
noResultUtilityForWhite = 0.0
drawEquivalentWinsForWhite = 0.5
cpuctExploration = 1.0
cpuctExplorationLog = 0.45
rootEndingBonusPoints = 0.5
rootPruneUselessMoves = true
useGraphSearch = true

In [ ]:
!cat ./katago/gtp.cfg

## 3. ベンチマーク (Benchmark - Optional)

動作確認のためにベンチマークを実行します。（スキップ可能）

In [ ]:
# @title ベンチマーク実行 (Run Benchmark)
# gtp.cfg の設定値のまま性能を計測する
!./katago/katago benchmark -model ./katago/{model_file_name} -config ./katago/gtp.cfg

## 4. 設定の自動調整 (Auto-tune - Optional)

最適なスレッド数 (numSearchThreads) を探索します。（スキップ可能）


In [ ]:
# @title 最適スレッド数の探索 (Search Optimal Threads)
# 最適な numSearchThreads を探索して表示します（gtp.cfg 自体は書き換えません）。
# 表示された推奨値を「2. 設定ファイルの作成」セルの numSearchThreads に反映して再実行してください。
# 設定ファイルそのものを対話形式で自動生成したい場合は genconfig コマンドが使えます:
#   !./katago/katago genconfig -model ./katago/{model_file_name} -output ./katago/gtp.cfg
!./katago/katago benchmark -config ./katago/gtp.cfg -model ./katago/{model_file_name} -tune


## 動作イメージ (Demo)


![GUI 動作の様子 / GUI demo](https://raw.githubusercontent.com/axyojp/katago-on-colab/main/img/gui-demo.png)

## 5. 実行 (Execution)

ここからKataGoを起動して対局や検討を行います。
実行するとログが表示され、KataGoがバックグラウンドで動作します。
盤面を表示している間は常時解析が動き続け、候補手と勝率は見ているだけで深まっていきます。
(その分 GPU は動き続けるため、使い終わったら「終了処理」を実行してください)


In [ ]:
# @title KataGoの起動 (Start KataGo)
import base64, subprocess, threading, queue, time, os, sys, json, re
from datetime import datetime
from IPython.display import display, HTML
from google.colab import output

# --- KataGo 設定 ---
KATAGO_PATH = "./katago/katago"
MODEL_PATH = f"./katago/{model_file_name}"
CONFIG_PATH = "./katago/gtp.cfg"
LOG_DIR = "./logs"

class KataGoGame:
    def __init__(self):
        self.current_turn = "Black"
        self.move_count = 0
        self.last_msg = "SYSTEM READY"
        self.win_rate = 50.0
        self.candidates = []
        self.total_visits = 0
        self.last_move = None
        self.move_history = []
        self.grid = [["." for _ in range(19)] for _ in range(19)]
        self.is_ready = False

        # --- ログ設定 ---
        # logフォルダを作成
        os.makedirs(LOG_DIR, exist_ok=True)
        # ファイル名を生成 (例: katago_20231027_123000.log)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.log_path = os.path.join(LOG_DIR, f"katago_{timestamp}.log")
        self.write_log(f"--- SESSION START: {timestamp} ---")

        # GPU 名は変わらないので起動時に一度だけ取得する
        self.gpu_name = None
        try:
            self.gpu_name = subprocess.run(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                capture_output=True, text=True, timeout=5).stdout.strip().splitlines()[0]
        except Exception:
            pass

        # エンジン起動
        self.proc = subprocess.Popen(
            [KATAGO_PATH, "gtp", "-model", MODEL_PATH, "-config", CONFIG_PATH],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True, bufsize=1
        )

        # 標準出力は専用スレッドで読み取り、キュー経由で受け渡す。
        # (select と readline の併用は readline が先読みした行を fd から消してしまい、
        #  以降の読み取りが常に1レスポンス遅れるため使わない)
        self.out_q = queue.Queue()
        threading.Thread(target=self._read_stdout, daemon=True).start()

        self.write_log("[LOG] KataGo起動中... (モデルロード完了まで待機)")
        # GTP エンジンは準備完了までコマンドに応答しないため、
        # version コマンドの応答をもって起動完了とみなす
        res = self.send_and_wait("version", timeout=180)
        if not any(r.startswith("=") for r in res):
            self.write_log("[ERROR] KataGo が 180 秒以内に応答しませんでした")
            raise RuntimeError(f"KataGo の起動に失敗しました。ログを確認してください: {self.log_path}")
        self.write_log(f"[LOG] KataGo起動完了 (version: {res[0].lstrip('= ')})")

        # 初期化
        self.send_and_wait("boardsize 19")
        self.send_and_wait("komi 6.5")
        self.send_and_wait("clear_board")
        self.refresh_board()
        self.is_ready = True
        # 常時解析を開始 (盤面を見ている間も GPU で解析が深まり続ける)
        self.start_analysis()
        self.write_log("[LOG] 全ての準備が整いました")
        print(f"KataGo Ready. Logs are saved in: {self.log_path}") # 完了だけ画面に通知

    def _read_stdout(self):
        for line in self.proc.stdout:
            self.out_q.put(line.rstrip("\n"))

    def _drain_output(self):
        while True:
            try: self.out_q.get_nowait()
            except queue.Empty: break

    def write_log(self, msg):
        """ログファイルに書き出すメソッド"""
        t_str = datetime.now().strftime('%H:%M:%S')
        try:
            with open(self.log_path, "a", encoding="utf-8") as f:
                f.write(f"[{t_str}] {msg}\n")
        except Exception as e:
            # ログ書き込み失敗時は標準エラーに出しておく
            sys.stderr.write(f"Log Error: {e}\n")

    def send_and_wait(self, cmd, timeout=2.0):
        """GTPコマンドを送信し、ログファイルに出力しながら応答を待つ"""
        self.write_log(f"GTP SEND > {cmd}") # ★ファイル出力に変更
        self.proc.stdin.write(cmd + "\n")
        self.proc.stdin.flush()

        res_lines = []
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                line = self.out_q.get(timeout=0.1).strip()
            except queue.Empty:
                continue
            if not line: continue
            self.write_log(f"GTP RECV < {line}")
            res_lines.append(line)
            if line.startswith("=") or line.startswith("?"):
                break
        return res_lines

    def start_analysis(self):
        """常時解析を開始する。次に GTP コマンドを送るまで裏で回り続け、
        結果 (info move 行) はキューに溜まり続ける"""
        self._drain_output()
        # interval 30 = 0.3秒ごとに報告 / rootInfo = 現局面の総探索数も報告させる
        self.proc.stdin.write("kata-analyze interval 30 rootInfo true\n")
        self.proc.stdin.flush()

    def stop_analysis(self):
        """常時解析を停止し、閉じ応答を読み捨てる。
        空行は GTP でコマンド扱いされないため応答を生まないが、走行中の
        kata-analyze は終了する。報告が1件も出る前に終了すると閉じ応答が
        「=」単体で出力され、直後に送るコマンドの成否判定を狂わせるため、
        コマンド送信前には必ずこれを呼んでキューを掃除する"""
        self.proc.stdin.write("\n")
        self.proc.stdin.flush()
        time.sleep(0.2)
        self._drain_output()

    def poll_analysis(self):
        """キューに溜まった解析結果のうち最新 (=最も探索が深い) 行を反映する"""
        last_line = None
        while True:
            try:
                line = self.out_q.get_nowait()
            except queue.Empty:
                break
            if "info move" in line and "winrate" in line:
                last_line = line
        if last_line:
            self.parse_kata_analyze(last_line)

    def read_showboard(self):
        """showboard を送り、盤面の最下段 (行番号 1) まで読み切って返す。
        GTP 応答は = 行が先頭に来るため、= で打ち切る send_and_wait は使えない"""
        self._drain_output()
        self.write_log("GTP SEND > showboard")
        self.proc.stdin.write("showboard\n")
        self.proc.stdin.flush()
        lines = []
        deadline = time.time() + 2.0
        while time.time() < deadline:
            try:
                line = self.out_q.get(timeout=0.1)
            except queue.Empty:
                if lines: break
                continue
            if line.strip():
                lines.append(line.rstrip())
            if re.match(r'^\s*1\s', line):
                break
        return lines

    def parse_kata_analyze(self, line):
        # rootInfo (現局面全体の情報) を分離し、総探索数を取り出す
        if "rootInfo" in line:
            line, root_part = line.split("rootInfo", 1)
            parts = root_part.split()
            try:
                self.total_visits = int(parts[parts.index("visits") + 1])
            except (ValueError, IndexError):
                pass
        move_infos = line.split("info move")[1:]
        parsed = []
        for info in move_infos[:5]:
            parts = info.split()
            try:
                m = parts[0]
                wr = float(parts[parts.index("winrate")+1]) * 100
                vs = int(parts[parts.index("visits")+1]) if "visits" in parts else 0
                parsed.append({"move": m, "rate": wr, "visits": vs})
            except: continue

        # ★ここに追加: 勝率が高い順(降順)にソート
        if parsed:
            parsed.sort(key=lambda x: x['rate'], reverse=True)

            self.candidates = parsed
            first_wr = parsed[0]["rate"]
            if self.current_turn == "Black":
                self.win_rate = first_wr
            else:
                self.win_rate = 100.0 - first_wr

    def refresh_board(self):
        """showboard で盤面キャッシュを更新する。
        GTP コマンドを送ると常時解析が止まるため、盤面が変わった時だけ呼び、
        呼んだ後は start_analysis() で解析を再開すること"""
        grid = [["." for _ in range(19)] for _ in range(19)]
        lines = self.read_showboard()

        for line in lines:
            match = re.match(r'^\s*(\d+)\s+(.+)', line)
            if not match: continue

            row_num = int(match.group(1))
            if not (1 <= row_num <= 19): continue

            board_str = match.group(2)
            r_idx = 19 - row_num
            stones = re.findall(r'[.XO@#]', board_str)

            for c_idx, s in enumerate(stones):
                if c_idx < 19:
                    if s in ['X', '#']: grid[r_idx][c_idx] = 'X'
                    elif s in ['O', '@']: grid[r_idx][c_idx] = 'O'
                    else: grid[r_idx][c_idx] = '.'

        self.grid = grid
        # 前の局面の候補手・探索数を消す (次の解析結果が届くまで Analyzing... 表示になる)
        self.candidates = []
        self.total_visits = 0

    def gpu_status(self):
        """GPU 使用率とメモリ使用量を取得する (取得できない環境では None)"""
        try:
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=5).stdout.strip().splitlines()[0]
            util, mem = [int(x.strip()) for x in out.split(",")]
            return util, mem
        except Exception:
            return None, None

    def get_state(self):
        if not self.is_ready: return {"ready": False}
        # ここでは GTP コマンドを送らない (常時解析を止めないため)。
        # キューに溜まった最新の解析結果と、キャッシュ済みの盤面を返す
        self.poll_analysis()
        gpu_util, gpu_mem = self.gpu_status()
        return {
            "ready": True, "grid": self.grid, "turn": self.current_turn,
            "move_count": self.move_count, "msg": self.last_msg,
            "win_rate": self.win_rate, "candidates": self.candidates,
            "total_visits": self.total_visits, "last_move": self.last_move,
            "gpu_name": self.gpu_name, "gpu_util": gpu_util, "gpu_mem_mb": gpu_mem
        }

# インスタンス作成
game = KataGoGame()

def api_get_state():
    # JS 側でのアンエスケープ処理を不要にするため base64 で返す
    payload = json.dumps(game.get_state(), ensure_ascii=False)
    return base64.b64encode(payload.encode("utf-8")).decode("ascii")

def api_handle_move(v):
    # 解析を止めてキューを掃除してからコマンドを送り、最後に必ず解析を再開する
    try:
        game.stop_analysis()
        if v == 'undo':
            res = game.send_and_wait("undo")
            if any(r.startswith("=") for r in res):
                game.current_turn = "White" if game.current_turn == "Black" else "Black"
                game.move_count = max(0, game.move_count - 1)
                game.last_msg = "UNDO SUCCESS"
                if game.move_history:
                    game.move_history.pop()
                prev = game.move_history[-1] if game.move_history else None
                game.last_move = prev if prev != "pass" else None
                game.refresh_board()
            else:
                game.last_msg = "ERR: cannot undo"
                game.write_log(f"[ERROR] Undo failed: {res}")
        elif v == 'pass':
            game.send_and_wait(f"play {game.current_turn.lower()} pass")
            game.current_turn = "White" if game.current_turn == "Black" else "Black"
            game.move_count += 1
            game.last_msg = "PASSED"
            game.move_history.append("pass")
            game.last_move = None
            game.refresh_board()
        else:
            cmd = f"play {game.current_turn.lower()} {v}"
            res = game.send_and_wait(cmd)

            if any(r.startswith("=") for r in res):
                game.current_turn = "White" if game.current_turn == "Black" else "Black"
                game.move_count += 1
                game.last_msg = f"MOVE: {v}"
                game.move_history.append(v)
                game.last_move = v
                game.refresh_board()
            else:
                err_line = next((r for r in res if r.startswith("?")), "? Unknown Error")
                err_msg = err_line[1:].strip()
                game.last_msg = f"ERR: {err_msg}"
                # printの代わりにファイルログへ出力
                game.write_log(f"[ERROR] Move failed: {v}, Reason: {err_msg}")
    finally:
        game.start_analysis()

GTP_COLS = "ABCDEFGHJKLMNOPQRST"

def _sgf_moves(sgf_text):
    """SGF のメインラインの着手を (色, GTP座標) のリストで返す。
    メインラインはテキスト上で最初の閉じ括弧までに現れる (変化はその後) ため、
    プロパティ値内の括弧とエスケープをスキップしながらそこまでを走査する"""
    main_chars = []
    in_value = False
    i = 0
    while i < len(sgf_text):
        ch = sgf_text[i]
        if in_value:
            if ch == "\\":
                main_chars.append(sgf_text[i:i+2]); i += 2; continue
            if ch == "]":
                in_value = False
        elif ch == "[":
            in_value = True
        elif ch == ")":
            break
        main_chars.append(ch)
        i += 1
    main = "".join(main_chars)
    moves = []
    for m in re.finditer(r"(?<![A-Za-z])([BW])\[([a-t]{0,2})\]", main):
        color, coord = m.group(1), m.group(2)
        if not coord or coord == "tt":
            moves.append((color, "pass"))
        else:
            col, row = ord(coord[0]) - 97, ord(coord[1]) - 97
            moves.append((color, f"{GTP_COLS[col]}{19 - row}"))
    return moves

def api_export_sgf():
    """現局面までの棋譜を SGF (base64) で返す。失敗時は空文字列"""
    try:
        game.stop_analysis()
        res = game.send_and_wait("printsgf")
        line = next((r for r in res if r.startswith("=")), None)
        sgf_text = line[1:].strip() if line else ""
        if not sgf_text:
            game.last_msg = "ERR: export failed"
            game.write_log(f"[ERROR] printsgf failed: {res}")
            return ""
        game.last_msg = "SGF EXPORTED"
        return base64.b64encode(sgf_text.encode("utf-8")).decode("ascii")
    finally:
        game.start_analysis()

def api_import_sgf(b64_content):
    # 解析を止めてから loadsgf し、成功時のみ Python 側の状態を更新する
    try:
        game.stop_analysis()
        sgf_text = base64.b64decode(b64_content).decode("utf-8", errors="replace")
        sgf_path = os.path.join(LOG_DIR, "imported.sgf")
        with open(sgf_path, "w", encoding="utf-8") as f:
            f.write(sgf_text)
        res = game.send_and_wait(f"loadsgf {sgf_path}", timeout=10)
        if any(r.startswith("=") for r in res):
            moves = _sgf_moves(sgf_text)
            game.move_history = [c for _, c in moves]
            game.move_count = len(moves)
            if moves:
                game.current_turn = "White" if moves[-1][0] == "B" else "Black"
                game.last_move = moves[-1][1] if moves[-1][1] != "pass" else None
            else:
                # 置き石だけの SGF は白番から、それ以外は黒番から
                game.current_turn = "White" if re.search(r"(?<![A-Za-z])AB\[", sgf_text) else "Black"
                game.last_move = None
            game.win_rate = 50.0
            game.last_msg = f"SGF LOADED ({game.move_count} moves)"
            game.refresh_board()
        else:
            err_line = next((r for r in res if r.startswith("?")), "? Unknown Error")
            game.last_msg = f"ERR: {err_line[1:].strip()}"
            game.write_log(f"[ERROR] loadsgf failed: {res}")
    finally:
        game.start_analysis()

def api_new_game():
    # 盤を初期化して新しい対局を開始する。成功時のみ Python 側の状態をリセットする
    try:
        game.stop_analysis()
        res = game.send_and_wait("clear_board")
        if any(r.startswith("=") for r in res):
            game.current_turn = "Black"
            game.move_count = 0
            game.move_history = []
            game.last_move = None
            game.win_rate = 50.0
            game.candidates = []
            game.total_visits = 0
            game.last_msg = "NEW GAME"
            game.refresh_board()
        else:
            game.last_msg = "ERR: new game failed"
            game.write_log(f"[ERROR] clear_board failed: {res}")
    finally:
        game.start_analysis()

output.register_callback('api_get_state', api_get_state)
output.register_callback('api_handle_move', api_handle_move)
output.register_callback('api_export_sgf', api_export_sgf)
output.register_callback('api_import_sgf', api_import_sgf)
output.register_callback('api_new_game', api_new_game)




HTML_UI = """
<div id="gui-box" style="display:flex; padding:20px; background:#1a2634; color:#ecf0f1; border-radius:12px; width:fit-content; font-family: sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.5);">
    <div style="position:relative;">
        <canvas id="goCanvas" width="460" height="460" style="background:#dcb35c; border:4px solid #2c3e50; cursor:pointer; border-radius:4px;"></canvas>
    </div>

    <div style="margin-left:25px; width:280px; display:flex; flex-direction:column; gap:12px;">
        <div style="background:#2c3e50; padding:15px; border-radius:8px;">
            <div id="info-count" style="font-size:0.8em; color:#95a5a6;">0 MOVES</div>
            <div id="info-turn" style="font-size:1.2em; font-weight:bold; margin:4px 0;">NEXT: <span style="display:inline-block; width:0.75em; height:0.75em; border-radius:50%; vertical-align:-0.05em; background:#111; border:1px solid #ecf0f1;"></span> BLACK</div>
            <div id="info-visits" style="font-size:0.8em; color:#f1c40f; margin:2px 0;">0 visits</div>
            <div id="info-msg" style="color:#3498db; font-family:monospace; font-size:0.8em; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;">SYSTEM READY</div>
            <div id="info-gpu" style="font-size:0.75em; color:#95a5a6; margin-top:6px; border-top:1px solid #455a64; padding-top:6px;">GPU: -</div>
        </div>

        <div style="background:#2c3e50; padding:15px; border-radius:8px;">
            <div style="display:flex; justify-content:space-between; font-size:0.8em; margin-bottom:8px; color:#bdc3c7;">
                <span>WIN RATE (BLACK)</span>
                <span id="wr-text">50.0%</span>
            </div>
            <div style="width:100%; background:#ecf0f1; height:14px; border-radius:7px; overflow:hidden; border:1px solid #455a64;">
                <div id="wr-bar" style="width:50%; background:#111; height:100%; transition: width 0.5s;"></div>
            </div>
        </div>

        <div style="background:#2c3e50; padding:15px; border-radius:8px; flex-grow:1;">
            <div style="font-size:0.8em; border-bottom:1px solid #455a64; padding-bottom:6px; margin-bottom:10px; color:#bdc3c7;">TOP CANDIDATES</div>
            <div id="cand-list" style="font-family:monospace; font-size:0.85em; display:flex; flex-direction:column; gap:6px;"></div>
        </div>

        <div style="display:grid; grid-template-columns: 1fr 1fr; gap:10px;">
            <button onclick="uiCmd('undo')" style="padding:10px; cursor:pointer; background:#34495e; color:white; border:none; border-radius:6px; font-weight:bold;">UNDO</button>
            <button onclick="uiCmd('pass')" style="padding:10px; cursor:pointer; background:#34495e; color:white; border:none; border-radius:6px; font-weight:bold;">PASS</button>
            <button onclick="uiSaveSgf()" style="padding:10px; cursor:pointer; background:#34495e; color:white; border:none; border-radius:6px; font-weight:bold;">SAVE SGF</button>
            <button onclick="document.getElementById('sgf-file').click()" style="padding:10px; cursor:pointer; background:#34495e; color:white; border:none; border-radius:6px; font-weight:bold;">LOAD SGF</button>
            <button onclick="uiNewGame()" style="grid-column:1 / -1; padding:10px; cursor:pointer; background:#7b3a34; color:white; border:none; border-radius:6px; font-weight:bold;">NEW GAME</button>
            <input type="file" id="sgf-file" accept=".sgf" style="display:none;" onchange="uiLoadSgf(this)">
        </div>
    </div>
</div>

<script>
(function() {
    const canvas = document.getElementById('goCanvas');
    const ctx = canvas.getContext('2d');
    const cols = "ABCDEFGHJKLMNOPQRST";
    let isRequesting = false;

  async function sync() {
        if(isRequesting) return;
        isRequesting = true;
        try {
            const result = await google.colab.kernel.invokeFunction('api_get_state', [], {});
            // Python 側は base64 文字列を返す (repr のクォートだけ除去してデコード)
            const rawData = result.data['text/plain'].replace(/^['"]|['"]$/g, '');
            const bytes = Uint8Array.from(atob(rawData), ch => ch.charCodeAt(0));
            const data = JSON.parse(new TextDecoder().decode(bytes));
            if(data && data.ready) draw(data);
        } catch(e) { console.error(e); }
        isRequesting = false;
    }
    setInterval(sync, 1500);

    function draw(data) {
        const pad = 30, size = 400, step = size/18;
        ctx.clearRect(0,0,460,460);

        // --- 盤面描画 ---
        ctx.textAlign = "center"; ctx.textBaseline = "middle";
        ctx.fillStyle = "#2c3e50"; ctx.font = "bold 12px sans-serif";
        for(let i=0; i<19; i++) {
            let p = pad + i*step;
            ctx.strokeStyle = "#2c3e50"; ctx.lineWidth = 1;
            ctx.beginPath(); ctx.moveTo(p, pad); ctx.lineTo(p, pad+size); ctx.stroke();
            ctx.beginPath(); ctx.moveTo(pad, p); ctx.lineTo(pad+size, p); ctx.stroke();
            ctx.fillText(cols[i], p, pad - 15); ctx.fillText(cols[i], p, pad + size + 15);
            let rowNum = 19 - i;
            ctx.fillText(rowNum, pad - 15, p); ctx.fillText(rowNum, pad + size + 15, p);
        }
        ctx.fillStyle = "#2c3e50";
        [3, 9, 15].forEach(r => [3, 9, 15].forEach(c => {
            ctx.beginPath(); ctx.arc(pad+c*step, pad+r*step, 3, 0, 7); ctx.fill();
        }));

        data.grid.forEach((row, r) => {
            row.forEach((s, c) => {
                if(s === 'X' || s === 'O') {
                    ctx.beginPath(); ctx.arc(pad+c*step, pad+r*step, step*0.45, 0, 7);
                    ctx.fillStyle = (s === 'X') ? "#000" : "#fff"; ctx.fill();
                    ctx.strokeStyle = (s === 'X') ? "#000" : "#2c3e50"; ctx.lineWidth = 1; ctx.stroke();
                }
            });
        });

        // 最後に打たれた手に赤いマーカーを付ける
        if (data.last_move) {
            const lmCol = cols.indexOf(data.last_move.charAt(0).toUpperCase());
            const lmRow = 19 - parseInt(data.last_move.substring(1));
            if (lmCol >= 0 && lmRow >= 0 && lmRow < 19) {
                ctx.beginPath(); ctx.arc(pad+lmCol*step, pad+lmRow*step, step*0.16, 0, 7);
                ctx.fillStyle = "#e74c3c"; ctx.fill();
            }
        }

        // --- 候補手の描画 (対数/指数的強調ロジック) ---
        if (data.candidates && data.candidates.length > 0) {
            // 候補手の中で最高の勝率を取得
            const maxRate = data.candidates.reduce((max, c) => Math.max(max, c.rate), 0);

            data.candidates.forEach((cand, index) => {
                if (cand.move === 'pass') return;

                const colChar = cand.move.charAt(0).toUpperCase();
                const rowStr = cand.move.substring(1);
                const rowNum = parseInt(rowStr);
                const cIdx = cols.indexOf(colChar);
                const rIdx = 19 - rowNum;

                if (cIdx >= 0 && rIdx >= 0 && rIdx < 19) {
                    const cx = pad + cIdx * step;
                    const cy = pad + rIdx * step;

                    // --- 強調ロジック ---
                    // 最高勝率に対する比率 (0.0 ~ 1.0)
                    const ratio = (maxRate > 0) ? (cand.rate / maxRate) : 0;

                    // 比率をべき乗して差を強調する (Gamma補正の逆のような処理)
                    // 値が1.0から少しでも下がると、急激に強調値が下がる
                    // 例: 比率0.9 (10%差) -> 0.9^6 = 0.53 (約50%の差に見える)
                    const emphasized = Math.pow(ratio, 6);

                    // 色相 (Hue): 0(赤) -> 60(黄)
                    // emphasized 1.0(最善) -> Hue 0 (真っ赤)
                    // emphasized 0.0(悪手) -> Hue 60 (黄色)
                    const hue = (1.0 - emphasized) * 60;

                    // 不透明度 (Alpha): 0.3(薄い) ~ 0.85(濃い)
                    // 最善手は濃く、悪い手は透ける
                    const alpha = 0.3 + (emphasized * 0.55);

                    // サイズ: 最善手は大きく、悪い手は少し小さく
                    const radius = (step * 0.35) + (emphasized * step * 0.1);

                    ctx.beginPath();
                    ctx.arc(cx, cy, radius, 0, 7);

                    // HSLAカラー設定
                    ctx.fillStyle = `hsla(${hue}, 100%, 50%, ${alpha})`;
                    ctx.fill();

                    // 白い縁取り (最善手付近のみ強くする)
                    if (emphasized > 0.5) {
                        ctx.strokeStyle = `rgba(255,255,255, ${0.4 + emphasized * 0.4})`;
                        ctx.lineWidth = 1.5;
                        ctx.stroke();
                    }

                    // テキスト (薄い黄色の円では黒文字にして読みやすくする)
                    ctx.fillStyle = (emphasized > 0.5) ? "#fff" : "#222";
                    ctx.font = "bold 10px sans-serif";
                    if (emphasized > 0.5) { ctx.shadowColor = "rgba(0,0,0,0.5)"; ctx.shadowBlur = 3; }
                    ctx.fillText(Math.round(cand.rate), cx, cy);
                    ctx.shadowBlur = 0;
                }
            });
        }
        // -----------------------------------------

        // UI更新
        document.getElementById('info-count').innerText = data.move_count + " MOVES";
        // 手番の石はパネルの文字色に依存させず明示的に塗る (暗色背景では ● が白く見えてしまうため)
        const stoneStyle = "display:inline-block; width:0.75em; height:0.75em; border-radius:50%; vertical-align:-0.05em;";
        const stone = (data.turn === "Black")
            ? `<span style="${stoneStyle} background:#111; border:1px solid #ecf0f1;"></span>`
            : `<span style="${stoneStyle} background:#fff; border:1px solid #111;"></span>`;
        document.getElementById('info-turn').innerHTML = "NEXT: " + stone + " " + (data.turn === "Black" ? "BLACK" : "WHITE");
        document.getElementById('info-msg').innerText = data.msg;
        document.getElementById('info-visits').innerText = (data.total_visits || 0).toLocaleString() + " visits";
        if (data.gpu_name) {
            // 使用率が高い = 解析が働いている状態なので緑で示す
            const utilColor = (data.gpu_util >= 30) ? "#2ecc71" : "#e67e22";
            const util = (data.gpu_util == null) ? "-" : data.gpu_util + "%";
            const mem = (data.gpu_mem_mb == null) ? "" : ` / ${data.gpu_mem_mb} MiB`;
            document.getElementById('info-gpu').innerHTML =
                `${data.gpu_name} ・ <span style="color:${utilColor}; font-weight:bold;">${util}</span>${mem}`;
        }
        document.getElementById('wr-text').innerText = data.win_rate.toFixed(1) + "%";
        document.getElementById('wr-bar').style.width = data.win_rate + "%";

        // リスト表示も同じ色ロジックで連動
        let html = "";
        const maxRateList = data.candidates.reduce((max, c) => Math.max(max, c.rate), 0);
        data.candidates.forEach((c, i) => {
            const ratio = (maxRateList > 0) ? (c.rate / maxRateList) : 0;
            const emphasized = Math.pow(ratio, 6);
            const hue = (1.0 - emphasized) * 60;
            const alpha = 0.5 + (emphasized * 0.5);
            // CSS HSL文字列
            const colorStyle = `hsl(${hue}, 100%, 62%)`;
            const opacityStyle = alpha;

            // 勝率・visits は固定幅+右揃えの列にして、桁数が違っても縦に揃うようにする
            html += `<div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px; opacity:${opacityStyle}">
                        <span style="color:${colorStyle}; font-weight:bold;">${i+1}. ${c.move}</span>
                        <span style="display:flex; align-items:baseline;">
                            <span style="color:#f1c40f; min-width:58px; text-align:right;">${c.rate.toFixed(1)}%</span>
                            <span style="color:#95a5a6; font-size:0.85em; min-width:64px; text-align:right;">${(c.visits||0).toLocaleString()}</span>
                        </span>
                     </div>`;
        });
        document.getElementById('cand-list').innerHTML = html || "Analyzing...";
    }

    canvas.onclick = async function(e) {
        const rect = canvas.getBoundingClientRect();
        const pad = 30, size = 400, step = size/18;
        const x = Math.round((e.clientX - rect.left - pad) / step);
        const y = Math.round((e.clientY - rect.top - pad) / step);
        if(x >= 0 && x < 19 && y >= 0 && y < 19) {
            const coord = cols[x] + (19 - y);
            document.getElementById('info-msg').innerText = "WAITING...";
            await google.colab.kernel.invokeFunction('api_handle_move', [coord], {});
            sync();
        }
    };
    window.uiCmd = async (v) => {
        await google.colab.kernel.invokeFunction('api_handle_move', [v], {});
        sync();
    };
    window.uiSaveSgf = async () => {
        const result = await google.colab.kernel.invokeFunction('api_export_sgf', [], {});
        const rawData = result.data['text/plain'].replace(/^['"]|['"]$/g, '');
        if (!rawData) { sync(); return; }
        const bytes = Uint8Array.from(atob(rawData), ch => ch.charCodeAt(0));
        const blob = new Blob([bytes], {type: "application/x-go-sgf"});
        const a = document.createElement('a');
        a.href = URL.createObjectURL(blob);
        a.download = "katago-" + new Date().toISOString().slice(0, 19).replace(/[T:]/g, "-") + ".sgf";
        a.click();
        URL.revokeObjectURL(a.href);
        sync();
    };
    window.uiNewGame = async () => {
        if (!confirm("新しい対局を開始しますか？ 現在の盤面は消去されます。\\nStart a new game? The current position will be cleared.")) return;
        document.getElementById('info-msg').innerText = "STARTING NEW GAME...";
        await google.colab.kernel.invokeFunction('api_new_game', [], {});
        sync();
    };
    window.uiLoadSgf = async (input) => {
        const file = input.files[0];
        if (!file) return;
        const text = await file.text();
        // 非ASCII (日本語コメント等) も送れるよう UTF-8 バイト列を base64 にする
        const utf8 = new TextEncoder().encode(text);
        let bin = "";
        utf8.forEach(b => { bin += String.fromCharCode(b); });
        document.getElementById('info-msg').innerText = "LOADING SGF...";
        await google.colab.kernel.invokeFunction('api_import_sgf', [btoa(bin)], {});
        input.value = "";
        sync();
    };
    sync();
})();
</script>
"""
display(HTML(HTML_UI))

## 6. 終了処理 (Cleanup)

対局・検討が終わったら実行して、裏で動いている KataGo プロセスを掃除します。


In [ ]:
# @title 終了処理 (Terminate KataGo)
import psutil

def terminate_katago_processes():
    for proc in psutil.process_iter():
        try:
            # プロセス名またはコマンドライン引数に'katago'が含まれるプロセスを探す
            if "katago" in proc.name().lower() or "katago" in " ".join(proc.cmdline()).lower():
                print(f"[LOG] Terminating KataGo process: PID={proc.pid}, Name={proc.name()}")
                proc.terminate() # プロセスを終了する
                proc.wait(timeout=3) # 終了を待つ（最大3秒）
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
            pass # プロセスが存在しない、アクセス拒否、ゾンビプロセスなどの場合はスキップ

terminate_katago_processes()

# フロントエンドからのポーリング(Showboard)を停止させるために、
# グローバルの game オブジェクトの状態を更新する
if 'game' in globals() and game is not None:
    print("[LOG] Updating game object state to stopped.")
    game.is_ready = False
    if hasattr(game, 'proc') and game.proc:
        game.proc.poll() # 終了ステータスを更新

print("[LOG] KataGo processes terminated.")